This notebook process one-by-one the netcdf files in `data/SWOT_L3_Expert_v3_0`.

For each file it:
- checks if the corresponding file is already present in the directory `data/SWOT_L3_Expert_v3_0_imbalance`,
- if it is not present, it:
    - computes the $u$ and $v$ imbalance components,
    - creates a new dataset with the same dimensions and coordinates as the original one but containing only the variables ``u_imb`` and ``v_imb``,
    - stores it in the directory `data/SWOT_L3_Expert_v3_0_imbalance` using the original filename.

In [ ]:
import os
import glob
import warnings

import jax
jax.config.update("jax_enable_x64", True)
import xarray as xr

from jaxparrow.cyclogeostrophy import cyclogeostrophic_imbalance

In [ ]:
warnings.filterwarnings("ignore")

INPUT_DIR = "data/SWOT_L3_Expert_v3_0"
OUTPUT_DIR = "data/SWOT_L3_Expert_v3_0_cg_imbalance"

os.makedirs(OUTPUT_DIR, exist_ok=True)

for filepath in sorted(glob.glob(os.path.join(INPUT_DIR, "*.nc"))):
    filename = os.path.basename(filepath)
    output_path = os.path.join(OUTPUT_DIR, filename)

    if os.path.exists(output_path):
        pass
    else:
        try:
            ds = xr.open_dataset(filepath)

            ug = ds.ugos_filtered.values
            vg = ds.vgos_filtered.values
            lat = ds.latitude.values
            lon = ds.longitude.values
            uimb, vimb = cyclogeostrophic_imbalance(ug, vg, ug, vg, lat, lon)

            ds_out = xr.Dataset(
                {
                    "uimb": xr.DataArray(uimb, dims=ds.ugos_filtered.dims, coords=ds.ugos_filtered.coords),
                    "vimb": xr.DataArray(vimb, dims=ds.vgos_filtered.dims, coords=ds.vgos_filtered.coords),
                },
            )
            ds_out.to_netcdf(output_path)

            ds.close()
        except Exception:
            print(f"Error for file {filename}")